# Animals-10 Local CPU Training

This notebook runs the full Animals-10 PyTorch flow locally on CPU. It does not use Google Colab, Google Drive, or `/content`. Outputs are written under this project folder.

Target CPU setting: AMD Ryzen 7 8845HS, 8 cores / 16 threads.


## Local Setup

Run this notebook from the `animals-10_classification` folder. If Jupyter starts from another directory, the setup cell falls back to the absolute project path on this laptop.


In [1]:
from pathlib import Path
import os
import platform
import sys

FALLBACK_WORKDIR = Path('/home/pah/fpga_cnn_accelerator/FPGA-based-CNN-Accelerator-for-autonomous-vehicles/CNN_model/python/animals-10_classification')
cwd = Path.cwd().resolve()
WORKDIR = cwd if (cwd / 'custom_cnn').is_dir() else FALLBACK_WORKDIR.resolve()
if not (WORKDIR / 'custom_cnn').is_dir():
    raise FileNotFoundError(f'Could not find animals-10_classification project folder from {cwd}')

os.chdir(WORKDIR)
os.environ['WORKDIR'] = str(WORKDIR)

# Leave a few hardware threads available for dataloader workers and the OS.
CPU_THREADS = 12
INTEROP_THREADS = 2
NUM_WORKERS = 4

os.environ['OMP_NUM_THREADS'] = str(CPU_THREADS)
os.environ['MKL_NUM_THREADS'] = str(CPU_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(CPU_THREADS)
os.environ['NUMEXPR_NUM_THREADS'] = str(CPU_THREADS)
os.environ['NUM_WORKERS'] = str(NUM_WORKERS)

os.environ['TEACHER_BATCH'] = '32'
os.environ['TEACHER_EVAL_BATCH'] = '64'
os.environ['STUDENT_BATCH'] = '128'
os.environ['STUDENT_EVAL_BATCH'] = '256'
os.environ['DISTILL_BATCH'] = '32'
os.environ['DISTILL_EVAL_BATCH'] = '128'

print('WORKDIR =', WORKDIR)
print('python =', sys.executable)
print('platform =', platform.platform())
print('cpu_count =', os.cpu_count())
print('torch_threads =', CPU_THREADS)
print('dataloader_workers =', NUM_WORKERS)


WORKDIR = /home/pah/fpga_cnn_accelerator/FPGA-based-CNN-Accelerator-for-autonomous-vehicles/CNN_model/python/animals-10_classification
python = /home/pah/fpga_cnn_accelerator/FPGA-based-CNN-Accelerator-for-autonomous-vehicles/CNN_model/python/animals-10_classification/.venv/bin/python
platform = Linux-6.8.0-94-generic-x86_64-with-glibc2.39
cpu_count = 16
torch_threads = 12
dataloader_workers = 4


## Dependencies and CPU Threads

Use the same virtual environment/kernel you use for this project. This cell installs missing packages into the active notebook kernel and confirms CPU execution.


In [2]:
%cd $WORKDIR
!python -m pip install -q -r requirements.txt

import torch
torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(INTEROP_THREADS)
except RuntimeError as exc:
    print('keeping existing interop thread setting:', exc)

print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
print('torch_num_threads:', torch.get_num_threads())
print('torch_num_interop_threads:', torch.get_num_interop_threads())


/home/pah/fpga_cnn_accelerator/FPGA-based-CNN-Accelerator-for-autonomous-vehicles/CNN_model/python/animals-10_classification
torch: 2.12.0+cu130
cuda_available: False
torch_num_threads: 12
torch_num_interop_threads: 2


## Dataset

The notebook uses the local dataset first: `data/animals10/raw-img`. If it is missing, it downloads the Kaggle zip using `download_animals10.sh` and extracts it locally.


In [3]:
from pathlib import Path
import os
import subprocess

data_root = WORKDIR / 'data' / 'animals10'
if not (data_root / 'raw-img').is_dir():
    print('Local Animals-10 data not found. Downloading from Kaggle...')
    subprocess.run(['bash', 'download_animals10.sh', '/tmp/animals10.zip', 'data/animals10'], check=True)

if not (data_root / 'raw-img').is_dir():
    raise RuntimeError(f'raw-img still not found under {data_root}')

os.environ['ANIMALS10_DATA_ROOT'] = str(data_root)
print('ANIMALS10_DATA_ROOT =', data_root)
print('class folders =', sorted(p.name for p in (data_root / 'raw-img').iterdir() if p.is_dir())[:12])


ANIMALS10_DATA_ROOT = /home/pah/fpga_cnn_accelerator/FPGA-based-CNN-Accelerator-for-autonomous-vehicles/CNN_model/python/animals-10_classification/data/animals10
class folders = ['cane', 'cavallo', 'elefante', 'farfalla', 'gallina', 'gatto', 'mucca', 'pecora', 'ragno', 'scoiattolo']


## Train EfficientNetB0 Teacher on CPU

This is slow on CPU because EfficientNetB0 runs at 224x224. Let it finish once, then use the saved `best.pt` for distillation.


In [4]:
%cd $WORKDIR
!python efficientnet/train_teacher.py \
  --data-root "$ANIMALS10_DATA_ROOT" \
  --manifest animals10_split_manifest.csv \
  --make-split \
  --backbone b0 \
  --out-dir efficientnet/runs/teacher_b0_cpu \
  --device cpu \
  --head-epochs 8 \
  --finetune-epochs 12 \
  --batch-size "$TEACHER_BATCH" \
  --eval-batch-size "$TEACHER_EVAL_BATCH" \
  --num-workers "$NUM_WORKERS"


/home/pah/fpga_cnn_accelerator/FPGA-based-CNN-Accelerator-for-autonomous-vehicles/CNN_model/python/animals-10_classification
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /home/pah/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100.0%
EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation

## Train Baseline RTL-Compatible Student A on CPU

This is the hardware-friendly student used for RTL export after BatchNorm folding.


In [5]:
%cd $WORKDIR
!python custom_cnn/train_student.py \
  --data-root "$ANIMALS10_DATA_ROOT" \
  --manifest animals10_split_manifest.csv \
  --variant A \
  --out-dir custom_cnn/runs/student_a_cpu \
  --device cpu \
  --epochs 60 \
  --batch-size "$STUDENT_BATCH" \
  --eval-batch-size "$STUDENT_EVAL_BATCH" \
  --num-workers "$NUM_WORKERS"


/home/pah/fpga_cnn_accelerator/FPGA-based-CNN-Accelerator-for-autonomous-vehicles/CNN_model/python/animals-10_classification
StudentCNNVersionA(
  (features): Sequential(
    (b1_conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (b1_bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (b1_relu1): ReLU()
    (b1_conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (b1_bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (b1_relu2): ReLU()
    (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (b2_conv1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (b2_bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (b2_relu1): ReLU()
    (b2_conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1

## Distill EfficientNetB0 Teacher into Student A on CPU

This keeps the final student graph RTL-compatible. The teacher is used only during training.


In [6]:
%cd $WORKDIR
!python custom_cnn/train_student_distill.py \
  --data-root "$ANIMALS10_DATA_ROOT" \
  --manifest animals10_split_manifest.csv \
  --teacher-checkpoint efficientnet/runs/teacher_b0_cpu/best.pt \
  --teacher-backbone b0 \
  --student-init custom_cnn/runs/student_a_cpu/best.pt \
  --variant A \
  --out-dir custom_cnn/runs/student_a_distill_b0_cpu \
  --device cpu \
  --epochs 40 \
  --batch-size "$DISTILL_BATCH" \
  --eval-batch-size "$DISTILL_EVAL_BATCH" \
  --temperature 4.0 \
  --ce-weight 0.50 \
  --num-workers "$NUM_WORKERS"


/home/pah/fpga_cnn_accelerator/FPGA-based-CNN-Accelerator-for-autonomous-vehicles/CNN_model/python/animals-10_classification
StudentCNNVersionA(
  (features): Sequential(
    (b1_conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (b1_bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (b1_relu1): ReLU()
    (b1_conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (b1_bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (b1_relu2): ReLU()
    (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (b2_conv1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (b2_bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (b2_relu1): ReLU()
    (b2_conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1

## Export Distilled Student INT8 Golden Files

This writes the RTL handoff package under `custom_cnn/int8_export`.


In [7]:
%cd $WORKDIR
!python custom_cnn/export_student_int8.py \
  --checkpoint custom_cnn/runs/student_a_distill_b0_cpu/best.pt \
  --data-root "$ANIMALS10_DATA_ROOT" \
  --manifest animals10_split_manifest.csv \
  --out-dir custom_cnn/int8_export \
  --device cpu \
  --num-workers "$NUM_WORKERS" \
  --balanced-test-cases-per-class 1

!python custom_cnn/reference_infer_int.py \
  --export-dir custom_cnn/int8_export \
  --case-index 0 \
  --dump-debug


/home/pah/fpga_cnn_accelerator/FPGA-based-CNN-Accelerator-for-autonomous-vehicles/CNN_model/python/animals-10_classification
bn_fold_max_abs_diff=0.00000924
exported_test_vectors=10 class_counts={0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1}
wrote INT8 export to custom_cnn/int8_export
prediction=0
final_logits_i8=34 6 -31 -59 -43 -19 11 6 -36 -62


## Show Final Metrics

Use this after training to quickly inspect the latest saved histories.


In [8]:
import json
from pathlib import Path

metric_files = [
    WORKDIR / 'efficientnet/runs/teacher_b0_cpu/metrics.json',
    WORKDIR / 'custom_cnn/runs/student_a_cpu/metrics.json',
    WORKDIR / 'custom_cnn/runs/student_a_distill_b0_cpu/metrics.json',
]

for path in metric_files:
    print('\n==', path.relative_to(WORKDIR), '==')
    if not path.exists():
        print('missing')
        continue
    history = json.loads(path.read_text())['history']
    for row in history[-3:]:
        print(row)



== efficientnet/runs/teacher_b0_cpu/metrics.json ==
{'accuracy': 0.9614992350841407, 'correct': 3771.0, 'epoch': 19, 'loss': 0.1230895184303412, 'phase': 'finetune', 'total': 3922.0, 'train_loss': 0.14655984695579935}
{'accuracy': 0.9625191228964813, 'correct': 3775.0, 'epoch': 20, 'loss': 0.12380939138096732, 'phase': 'finetune', 'total': 3922.0, 'train_loss': 0.13995384698272753}
{'accuracy': 0.963151207115629, 'correct': 3790.0, 'epoch': 20, 'loss': 0.1277272563706965, 'phase': 'test', 'total': 3935.0}

== custom_cnn/runs/student_a_cpu/metrics.json ==
{'accuracy': 0.8378378378378378, 'correct': 3286.0, 'epoch': 59, 'loss': 0.5082781410898134, 'phase': 'train', 'total': 3922.0, 'train_loss': 0.34098919902406344}
{'accuracy': 0.8391126976032637, 'correct': 3291.0, 'epoch': 60, 'loss': 0.5119509553495929, 'phase': 'train', 'total': 3922.0, 'train_loss': 0.3389488444417055}
{'accuracy': 0.8216010165184244, 'correct': 3233.0, 'epoch': 60, 'images_per_second': 1032.1179095653874, 'loss':